# EpiTOC3

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.EpiTOC3)

class EpiTOC3(epiTOC2):
    pass



In [3]:
model = pya.models.EpiTOC3()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'epitoc3'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Teschendorff, Andrew E. \"A comparison of epigenetic mitotic-like clocks for cancer risk prediction.\" Genome Medicine 12.1 (2020): 56."
model.metadata["doi"] = "https://doi.org/10.1186/s13073-020-00752-3"
model.metadata["research_only"] = None
model.metadata["notes"] = None

## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/epiTOC3.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
model.features = df['probe'].astype(str).tolist()

## Load weights into base model

In [7]:
model.delta = torch.tensor(df['delta'].values, dtype=torch.float32).unsqueeze(0)
model.beta0 = torch.tensor(df['beta0'].values, dtype=torch.float32).unsqueeze(0)

In [8]:
model.base_model = None

## Load reference values

In [9]:
model.reference_values = [-1]*len(model.features)

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "nan_to_zero"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Teschendorff, Andrew E. "A comparison of epigenetic mitotic-like '
             'clocks for cancer risk prediction." Genome Medicine 12.1 (2020): '
             '56.',
 'clock_name': 'epitoc3',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13073-020-00752-3',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]... [Total elements: 170]
preprocess_name: 'nan_to_zero'
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg23062112', 'cg18560328', 'cg21229268', 'cg11250773', 'cg12892303', 'cg12781700', 'cg12065366', 'cg17712694', 'cg12799781', 'cg01718742', 'cg10263370', '

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ 1921.6122],
        [-4698.6198],
        [ 1365.4274],
        [-4247.0676],
        [  -35.5126],
        [  116.7216],
        [ 7356.7134],
        [-3367.8507],
        [ 7780.6034],
        [-7170.8200]], dtype=torch.float64)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
